# LLM Trend Visualizer — Step 4 & 5: Embedding + Clustering

**运行前**：在 VSCode 资源管理器中将本地的 `papers_filtered.csv` 拖拽上传到 Colab 的 `/content/` 目录。

In [8]:
# Cell 1: 环境检查
import os, torch
print(f'GPU 可用：{torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU 型号：{torch.cuda.get_device_name(0)}')

# ⚠️ Colab 是远程服务器，本地 Mac 路径在这里无法访问
# 可以上传完整文件 papers_filtered.csv，
# 或上传拆分文件 papers_filtered1.csv + papers_filtered2.csv 到 /content/
DATA_DIR = '/content'
papers_path = f'{DATA_DIR}/papers_filtered.csv'
papers_part1_path = f'{DATA_DIR}/papers_filtered1.csv'
papers_part2_path = f'{DATA_DIR}/papers_filtered2.csv'

has_full = os.path.exists(papers_path)
has_split = os.path.exists(papers_part1_path) and os.path.exists(papers_part2_path)

assert has_full or has_split, (
    f'找不到输入文件：{papers_path} 或 split 文件\n'
    '请上传 papers_filtered.csv，或者上传 papers_filtered1.csv + papers_filtered2.csv 到 /content/'
)

if has_full:
    print(f'✅ 找到完整输入文件：{papers_path}')
else:
    print(f'✅ 找到拆分输入文件：{papers_part1_path} + {papers_part2_path}')



GPU 可用：True
GPU 型号：NVIDIA L4
✅ 找到拆分输入文件：/content/papers_filtered1.csv + /content/papers_filtered2.csv


In [2]:
# Cell 2: 安装依赖
!pip install -q umap-learn hdbscan sentence-transformers

In [3]:
# Cell 3: 读取数据
import numpy as np
import pandas as pd

if os.path.exists(papers_path):
    df = pd.read_csv(papers_path)
    print('使用完整 papers_filtered.csv')
else:
    df1 = pd.read_csv(papers_part1_path)
    df2 = pd.read_csv(papers_part2_path)
    df = pd.concat([df1, df2], ignore_index=True)
    print('使用拆分文件 papers_filtered1.csv + papers_filtered2.csv')

print(f'论文总数：{len(df)}')
print(df['venue'].value_counts())



使用拆分文件 papers_filtered1.csv + papers_filtered2.csv
论文总数：54620
venue
Findings of ACL    9546
EMNLP              8127
ACL                7382
NAACL              3266
LREC               2697
                   ... 
PAIL                  2
ETHNLP                1
MML                   1
UMRPW                 1
ICNLSP                1
Name: count, Length: 387, dtype: int64


使用拆分文件 papers_filtered1.csv + papers_filtered2.csv
论文总数：54620
venue
Findings of ACL    9546
EMNLP              8127
ACL                7382
NAACL              3266
LREC               2697
                   ... 
PAIL                  2
ETHNLP                1
MML                   1
UMRPW                 1
ICNLSP                1
Name: count, Length: 387, dtype: int64


In [9]:
# Cell 4: OpenRouter Embedding (qwen/qwen3-embedding-8b)
import requests, time
import numpy as np
import pandas as pd
from tqdm.notebook import tqdm

OPENROUTER_API_KEY = "YOUR_OPENROUTER_API_KEY_HERE"  # ← 在此填入你的 OpenRouter API Key
OPENROUTER_MODEL   = "qwen/qwen3-embedding-8b"
OPENROUTER_URL     = "https://openrouter.ai/api/v1/embeddings"
BATCH_SIZE         = 64

EMBEDDINGS_PATH = f'{DATA_DIR}/embeddings.npy'
IDS_CSV         = f'{DATA_DIR}/embedding_ids.csv'

assert OPENROUTER_API_KEY, "请先填入 OPENROUTER_API_KEY"

# 断点续传
done_ids = set()
if os.path.exists(EMBEDDINGS_PATH) and os.path.exists(IDS_CSV):
    done_ids = set(pd.read_csv(IDS_CSV)['paper_id'].tolist())
    print(f'续传：已完成 {len(done_ids)} 篇，剩余 {len(df) - len(done_ids)} 篇')

remaining   = df[~df['paper_id'].isin(done_ids)].copy()
texts       = (remaining['title'].fillna('') + ' [SEP] ' + remaining['abstract'].fillna('')).tolist()
paper_ids   = remaining['paper_id'].tolist()

headers = {'Authorization': f'Bearer {OPENROUTER_API_KEY}', 'Content-Type': 'application/json'}
batches = [texts[i:i+BATCH_SIZE] for i in range(0, len(texts), BATCH_SIZE)]

all_vecs = []
for batch in tqdm(batches, desc='OpenRouter embedding'):
    for attempt in range(3):
        try:
            resp = requests.post(
                OPENROUTER_URL,
                headers=headers,
                json={'model': OPENROUTER_MODEL, 'input': batch},
                timeout=60,
            )
            if resp.status_code == 429:
                wait = int(resp.headers.get('Retry-After', 15))
                print(f'Rate limit，等待 {wait}s...')
                time.sleep(wait)
                continue
            resp.raise_for_status()
            all_vecs.extend([item['embedding'] for item in resp.json()['data']])
            time.sleep(0.1)
            break
        except Exception as e:
            print(f'请求失败 (attempt {attempt+1}): {e}')
            time.sleep(5)

new_emb = np.array(all_vecs, dtype=np.float32)

if os.path.exists(EMBEDDINGS_PATH) and len(done_ids) > 0:
    all_emb     = np.vstack([np.load(EMBEDDINGS_PATH), new_emb])
    all_ids_df  = pd.concat([pd.read_csv(IDS_CSV), remaining[['paper_id']].reset_index(drop=True)], ignore_index=True)
else:
    all_emb     = new_emb
    all_ids_df  = remaining[['paper_id']].reset_index(drop=True)

np.save(EMBEDDINGS_PATH, all_emb)
all_ids_df.to_csv(IDS_CSV, index=False)
print(f'✅ Embedding 完成：shape={all_emb.shape}')

OpenRouter embedding:   0%|          | 0/854 [00:00<?, ?it/s]

✅ Embedding 完成：shape=(54620, 4096)


In [10]:
# Cell 5: UMAP 10D 降维（用于聚类）
import umap

print(f'embeddings shape: {all_emb.shape}')
print('UMAP 降维（→10D）...')

reducer_10d = umap.UMAP(
    n_components=10, n_neighbors=15, min_dist=0.0,
    metric='cosine', random_state=42, verbose=True,
)
reduced_10d = reducer_10d.fit_transform(all_emb)
print(f'完成：shape={reduced_10d.shape}')

embeddings shape: (54620, 4096)
UMAP 降维（→10D）...
UMAP(angular_rp_forest=True, metric='cosine', min_dist=0.0, n_components=10, n_jobs=1, random_state=42, verbose=True)


/usr/local/lib/python3.12/dist-packages/umap/umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(


Sat Mar 21 08:23:01 2026 Construct fuzzy simplicial set
Sat Mar 21 08:23:01 2026 Finding Nearest Neighbors
Sat Mar 21 08:23:01 2026 Building RP forest with 17 trees
Sat Mar 21 08:23:22 2026 NN descent for 16 iterations
	 1  /  16
	 2  /  16
	 3  /  16
	 4  /  16
	 5  /  16
	Stopping threshold met -- exiting after 5 iterations
Sat Mar 21 08:24:13 2026 Finished Nearest Neighbor Search
Sat Mar 21 08:24:16 2026 Construct embedding


Epochs completed:   0%|            0/200 [00:00]

	completed  0  /  200 epochs
	completed  20  /  200 epochs
	completed  40  /  200 epochs
	completed  60  /  200 epochs
	completed  80  /  200 epochs
	completed  100  /  200 epochs
	completed  120  /  200 epochs
	completed  140  /  200 epochs
	completed  160  /  200 epochs
	completed  180  /  200 epochs
Sat Mar 21 08:25:26 2026 Finished embedding
完成：shape=(54620, 10)


In [11]:
print(reduced_10d.shape)

(54620, 10)


In [12]:
np.save(f'{DATA_DIR}/reduced_embeddings.npy', reduced_10d)

In [5]:
import numpy as np
reduced_10d = np.load(f'{DATA_DIR}/reduced_embeddings.npy')

In [13]:
# Cell 6: 迭代式 HDBSCAN 聚类（80 -> 40 -> 20，逐步细分 Others）
import hdbscan
import numpy as np
import pandas as pd

MIN_CLUSTER_SIZES = [80, 40, 20]
MIN_SAMPLES = 5
TARGET_NOISE_RATIO = 0.10

def refine_noise_clusters(reduced_embeddings, min_cluster_sizes, min_samples=5, target_noise_ratio=0.10):
    labels = np.full(len(reduced_embeddings), -1, dtype=int)
    pending_idx = np.arange(len(reduced_embeddings))
    next_cluster_id = 0

    for round_id, min_cluster_size in enumerate(min_cluster_sizes, start=1):
        if len(pending_idx) == 0:
            break

        print(f"\n=== Round {round_id}: HDBSCAN min_cluster_size={min_cluster_size} on {len(pending_idx)} papers ===")
        clusterer = hdbscan.HDBSCAN(
            min_cluster_size=min_cluster_size,
            min_samples=min_samples,
            metric='euclidean',
            cluster_selection_method='eom',
        )
        round_labels = clusterer.fit_predict(reduced_embeddings[pending_idx])

        unique_labels = sorted(set(round_labels) - {-1})
        print(f"Found {len(unique_labels)} new clusters in this round")

        for local_label in unique_labels:
            mask = round_labels == local_label
            original_idx = pending_idx[mask]
            labels[original_idx] = next_cluster_id
            next_cluster_id += 1

        pending_idx = pending_idx[round_labels == -1]
        noise_ratio = len(pending_idx) / len(reduced_embeddings)
        print(f"Remaining Others: {len(pending_idx)} papers ({noise_ratio:.1%})")

        if noise_ratio <= target_noise_ratio:
            print(f"Noise ratio already <= {target_noise_ratio:.0%}, stop refining")
            break

    return labels

labels = refine_noise_clusters(
    reduced_10d,
    min_cluster_sizes=MIN_CLUSTER_SIZES,
    min_samples=MIN_SAMPLES,
    target_noise_ratio=TARGET_NOISE_RATIO,
)

n_clusters = len(set(labels)) - (1 if -1 in labels else 0)
noise_ratio = (labels == -1).sum() / len(labels)

print(f"\n最终结果：{n_clusters} 个 cluster，Others 占比 {noise_ratio:.1%}")
label_counts = pd.Series(labels).value_counts()
print("\nTop 20 cluster 大小：")
print(label_counts[label_counts.index != -1].head(20).to_string())
print(f"\nOthers 数量：{int((labels == -1).sum())}")



=== Round 1: HDBSCAN min_cluster_size=80 on 54620 papers ===


/usr/local/lib/python3.12/dist-packages/hdbscan/robust_single_linkage_.py:175: SyntaxWarning: invalid escape sequence '\{'
  $max \{ core_k(a), core_k(b), 1/\alpha d(a,b) \}$.


Found 161 new clusters in this round
Remaining Others: 14348 papers (26.3%)

=== Round 2: HDBSCAN min_cluster_size=40 on 14348 papers ===
Found 99 new clusters in this round
Remaining Others: 5759 papers (10.5%)

=== Round 3: HDBSCAN min_cluster_size=20 on 5759 papers ===
Found 68 new clusters in this round
Remaining Others: 2674 papers (4.9%)
Noise ratio already <= 10%, stop refining

最终结果：328 个 cluster，Others 占比 4.9%

Top 20 cluster 大小：
61     1551
23     1378
143    1285
120    1053
72      984
20      862
91      814
70      670
68      655
137     609
107     583
117     582
209     563
153     532
25      492
96      484
156     453
93      448
47      443
130     432

Others 数量：2674


In [14]:
# Cell 7: 保存聚类结果（不做可视化）
import numpy as np
import pandas as pd

LABELS_PATH = f'{DATA_DIR}/cluster_labels.npy'
CLUSTERED_CSV = f'{DATA_DIR}/papers_clustered.csv'

np.save(LABELS_PATH, labels)

all_ids_df = pd.read_csv(f'{DATA_DIR}/embedding_ids.csv')
all_ids_df['cluster_id'] = labels
result = df.merge(all_ids_df, on='paper_id', how='inner')
result.to_csv(CLUSTERED_CSV, index=False)

print('✅ 聚类结果已保存')
print(f'cluster_labels.npy -> {LABELS_PATH}')
print(f'papers_clustered.csv -> {CLUSTERED_CSV}')
print(f'rows: {len(result)}')


✅ 聚类结果已保存
cluster_labels.npy -> /content/cluster_labels.npy
papers_clustered.csv -> /content/papers_clustered.csv
rows: 54620


In [ ]:
# Cell 8: 结果检查
import pandas as pd

label_counts = pd.Series(labels).value_counts()
n_clusters = len(set(labels)) - (1 if -1 in labels else 0)
noise_ratio = (labels == -1).mean()

print(f'clusters: {n_clusters}')
print(f'Others ratio: {noise_ratio:.1%}')
print('\nCluster size distribution (top 30):')
print(label_counts.head(30).to_string())

if noise_ratio > 0.10:
    print('\n⚠️ Others 仍然超过 10%，建议下一轮尝试更激进参数（例如 min_samples=3）')
else:
    print('\n✅ Others 已控制在 10% 以内或接近目标')


In [ ]:
# Cell 9: 下载结果到本地
from google.colab import files
files.download(CLUSTERED_CSV)
files.download(LABELS_PATH)
